In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob
image_paths = glob.glob("/content/drive/MyDrive/**/*.jpg", recursive=True)
image_paths = [p for p in image_paths if "hard_negatives" not in p and "checkpoints" not in p]
print(f"Total fotos normales: {len(image_paths)}")

hard_neg_paths = glob.glob("/content/drive/MyDrive/hard_negatives/**/*.jpg", recursive=True)
hard_neg_paths += glob.glob("/content/drive/MyDrive/hard_negatives/**/*.png", recursive=True)
print(f"Total hard negatives: {len(hard_neg_paths)}")

In [ ]:
import numpy as np
import random
import cv2

def generate_dust_mask(
    image_shape=(1024, 1024),
    num_blobs=120,
    num_scratches=4,
    num_hairs=50,
    max_blob_size=3,
    max_scratch_length=240,
    max_hair_length=20,
    squiggliness=1,
    scale_factor=4
):
    h, w = image_shape
    H, W = h * scale_factor, w * scale_factor
    mask_hi = np.zeros((H, W), dtype=np.uint8)
    temp = np.zeros_like(mask_hi)

    def rand_count(base):
        return random.randint(int(base * 0.8), int(base * 1.2))

    num_blobs     = rand_count(num_blobs)
    num_scratches = rand_count(num_scratches)
    num_hairs     = rand_count(num_hairs)
    max_blob_size = max(1, max_blob_size + random.randint(-1, 2))

    for _ in range(num_blobs):
        temp.fill(0)
        center = (random.randint(0, W), random.randint(0, H))
        axes = (
            random.randint(max(1, max_blob_size - 1), max_blob_size + 1) * scale_factor,
            random.randint(1, max(1, max_blob_size // 2) + 1) * scale_factor
        )
        angle = random.randint(0, 180)
        cv2.ellipse(temp, center, axes, angle, 0, 360, 255, -1)
        opacity = random.uniform(0.3, 1.0)
        mask_hi = cv2.addWeighted(mask_hi, 1.0, temp, opacity, 0)

    for _ in range(num_scratches):
        temp.fill(0)
        x1, y1 = random.randint(0, W), random.randint(0, H)
        angle = random.uniform(0, 2 * np.pi)
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        length = random.randint(
            int(max_scratch_length * 0.7), int(max_scratch_length * 1.3)
        ) * scale_factor
        x2 = int(x1 + length * cos_a)
        y2 = int(y1 + length * sin_a)
        cv2.line(temp, (x1, y1), (x2, y2), 255, scale_factor)
        opacity = random.uniform(0.2, 1.0)
        mask_hi = cv2.addWeighted(mask_hi, 1.0, temp, opacity, 0)

    for _ in range(num_hairs):
        temp.fill(0)
        points = []
        x, y = random.randint(0, W), random.randint(0, H)
        hair_length = random.randint(
            int(max_hair_length * 0.8), int(max_hair_length * 1.2)
        ) * scale_factor
        num_segments = max(3, hair_length // random.randint(8, 12))
        hair_squiggliness = squiggliness * random.uniform(0.8, 1.2)
        angle = random.uniform(0, 2 * np.pi)
        dx_base, dy_base = np.cos(angle), np.sin(angle)
        for _ in range(num_segments):
            seg_len = random.uniform(8, 12) * scale_factor
            dx = dx_base * seg_len + hair_squiggliness * random.uniform(-seg_len, seg_len)
            dy = dy_base * seg_len + hair_squiggliness * random.uniform(-seg_len, seg_len)
            x = int(np.clip(x + dx, 0, W - 1))
            y = int(np.clip(y + dy, 0, H - 1))
            points.append((x, y))
        if len(points) >= 2:
            pts = np.array(points, dtype=np.int32)
            thickness = random.randint(1, 2) * scale_factor
            cv2.polylines(temp, [pts], isClosed=False, color=255,
                          thickness=thickness, lineType=cv2.LINE_AA)
            opacity = random.uniform(0.2, 1.0)
            mask_hi = cv2.addWeighted(mask_hi, 1.0, temp, opacity, 0)

    mask_lo = cv2.resize(mask_hi, (w, h), interpolation=cv2.INTER_AREA)
    return mask_lo.astype(np.float32) / 255.0


def apply_dust_to_greyscale_image(image_grey, mask, intensity=255):
    mask_float  = mask.astype(np.float32)
    image_float = image_grey.astype(np.float32)
    image_float += mask_float * intensity
    return np.clip(image_float, 0, 255).astype(np.uint8)

In [ ]:
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import os
import albumentations as A

transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.Rotate(limit=5, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=3, p=0.3)
])


class DustDataset(Dataset):
    def __init__(self, root_dir, image_size=(512, 512), transform=None):
        self.image_paths = glob.glob(os.path.join(root_dir, "**/*.jpg"), recursive=True)
        self.image_paths = [p for p in self.image_paths if "hard_negatives" not in p and "checkpoints" not in p]
        self.image_size  = image_size
        self.transform   = transform
        print(f"DustDataset: {len(self.image_paths)} imagenes")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        clean_img = cv2.imread(path)
        if clean_img is None:
            raise FileNotFoundError(f"No se pudo leer {path}")
        if len(clean_img.shape) == 3:
            clean_img = cv2.cvtColor(clean_img, cv2.COLOR_BGR2GRAY)
        clean_img = cv2.resize(clean_img, self.image_size)
        mask = generate_dust_mask(image_shape=self.image_size, squiggliness=random.uniform(0.5, 1.5))
        dust_intensity = random.randint(80, 255)
        dusty_img = apply_dust_to_greyscale_image(clean_img, mask, intensity=dust_intensity)
        image = dusty_img.astype(np.float32) / 255.0
        image = image[..., np.newaxis]
        mask  = mask.astype(np.float32)[..., np.newaxis]
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']
        image = np.transpose(image, (2, 0, 1))
        mask  = np.transpose(mask,  (2, 0, 1))
        return image, mask


class HardNegativeDataset(Dataset):
    def __init__(self, root_dir, image_size=(512, 512), transform=None, repeat_factor=5):
        self.image_paths = glob.glob(os.path.join(root_dir, "**/*.jpg"), recursive=True)
        self.image_paths += glob.glob(os.path.join(root_dir, "**/*.png"), recursive=True)
        self.image_size    = image_size
        self.transform     = transform
        self.repeat_factor = repeat_factor
        print(f"HardNegativeDataset: {len(self.image_paths)} imagenes x {repeat_factor} = {len(self)} ejemplos")

    def __len__(self):
        return len(self.image_paths) * self.repeat_factor

    def __getitem__(self, idx):
        path = self.image_paths[idx % len(self.image_paths)]
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(f"No se pudo leer {path}")
        if len(img.shape) == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.resize(img, self.image_size)
        image = img.astype(np.float32) / 255.0
        image = image[..., np.newaxis]
        mask  = np.zeros_like(image)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']
        image = np.transpose(image, (2, 0, 1))
        mask  = np.transpose(mask,  (2, 0, 1))
        return image, mask


HARD_NEGATIVES_DIR = "/content/drive/MyDrive/hard_negatives"
IMAGE_SIZE = (512, 512)

dust_dataset = DustDataset("/content/drive/MyDrive", image_size=IMAGE_SIZE, transform=transform)

if os.path.exists(HARD_NEGATIVES_DIR) and len(glob.glob(os.path.join(HARD_NEGATIVES_DIR, "**/*.*"), recursive=True)) > 0:
    hard_neg_dataset = HardNegativeDataset(HARD_NEGATIVES_DIR, image_size=IMAGE_SIZE, transform=transform, repeat_factor=5)
    combined_dataset = ConcatDataset([dust_dataset, hard_neg_dataset])
    print(f"\nDataset combinado: {len(dust_dataset)} normales + {len(hard_neg_dataset)} negativos = {len(combined_dataset)} total")
else:
    combined_dataset = dust_dataset
    print(f"\nSin hard_negatives — entrenando solo con datos normales ({len(dust_dataset)} imagenes)")

loader = DataLoader(combined_dataset, batch_size=4, shuffle=True, num_workers=2)

In [ ]:
import torch
import torch.nn as nn

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1), nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1), nn.ReLU()
            )
        self.enc1 = conv_block(1, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)
        self.enc4 = conv_block(256, 512)
        self.pool = nn.MaxPool2d(2)
        self.middle = conv_block(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = conv_block(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.final = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        m  = self.middle(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(m),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.final(d1))

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando device: {device}")

model     = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()

os.makedirs("/content/drive/MyDrive/checkpoints", exist_ok=True)

checkpoints = sorted(glob.glob("/content/drive/MyDrive/checkpoints/unet_epoch*.pth"))
if checkpoints:
    checkpoint_path = checkpoints[-1]
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    try:
        START_EPOCH = int(checkpoint_path.split("epoch")[-1].replace(".pth", "")) + 1
    except Exception:
        START_EPOCH = 0
    print(f"Checkpoint cargado: {checkpoint_path} — continuando desde epoch {START_EPOCH}")
else:
    START_EPOCH = 0
    print("Sin checkpoint — entrenando desde cero")

NUM_EPOCHS = 30

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss    = 0
    current_epoch = START_EPOCH + epoch

    for images, masks in tqdm(loader, desc=f"Epoch {current_epoch}"):
        images = images.to(device)
        masks  = masks.to(device)
        preds  = model(images)
        loss   = criterion(preds, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {current_epoch}: Loss = {avg_loss:.4f}")
    torch.save(model.state_dict(), f"/content/drive/MyDrive/checkpoints/unet_epoch{current_epoch:02d}.pth")
    print(f"Checkpoint guardado: epoch {current_epoch}")

print("\nEntrenamiento completado.")

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

def predict_dust_mask(model, weights_path, image_path, threshold=0.5, window_size=1024, stride=512, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model = model.to(device)
    model.eval()
    image = PILImage.open(image_path).convert('L')
    image_np = np.array(image).astype(np.float32)
    H, W = image_np.shape
    pad_h = (window_size - H % stride) % stride
    pad_w = (window_size - W % stride) % stride
    padded = np.pad(image_np, ((0, pad_h), (0, pad_w)), mode='reflect')
    pH, pW = padded.shape
    prediction_map = np.zeros((pH, pW), dtype=np.float32)
    count_map      = np.zeros((pH, pW), dtype=np.float32)
    with torch.no_grad():
        for y in range(0, pH - window_size + 1, stride):
            for x in range(0, pW - window_size + 1, stride):
                patch = padded[y:y+window_size, x:x+window_size]
                patch_tensor = torch.from_numpy(patch).float().unsqueeze(0).unsqueeze(0) / 255.0
                patch_tensor = patch_tensor.to(device)
                pred = model(patch_tensor)
                pred_np = pred.squeeze().cpu().numpy()
                prediction_map[y:y+window_size, x:x+window_size] += pred_np
                count_map[y:y+window_size, x:x+window_size]      += 1.0
    final_mask = prediction_map / np.maximum(count_map, 1e-8)
    final_mask = final_mask[:H, :W]
    return final_mask


def inpaint_dusty_image(dusty_image, dust_mask, threshold=0.25):
    binary_mask = (dust_mask > threshold).astype(np.uint8)
    kernel = np.ones((5, 5), np.uint8)
    binary_mask = cv2.dilate(binary_mask, kernel, iterations=1)
    return cv2.inpaint(dusty_image, binary_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)


checkpoints  = sorted(glob.glob("/content/drive/MyDrive/checkpoints/unet_epoch*.pth"))
weights_path = checkpoints[-1]
print(f"Usando checkpoint: {weights_path}")

image_path = "/content/jpg572.jpg"

model_inf = UNet()
dust = predict_dust_mask(model_inf, weights_path, image_path, threshold=0.2, window_size=1024, stride=980)

test_image = PILImage.open(image_path).convert('L')
test_np    = np.array(test_image)
result     = inpaint_dusty_image(test_np, dust, threshold=0.25)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1); plt.imshow(test_np, cmap='gray'); plt.title("Original"); plt.axis('off')
plt.subplot(1, 3, 2); plt.imshow(dust > 0.01, cmap='gray'); plt.title("Polvo detectado"); plt.axis('off')
plt.subplot(1, 3, 3); plt.imshow(result, cmap='gray'); plt.title("Resultado limpio"); plt.axis('off')
plt.tight_layout()
plt.show()

PILImage.fromarray(result).save("/content/drive/MyDrive/resultado_limpio.jpg")
print("Guardado en Drive")